# Reproducing a benchmark's published model score for aqueous solubility (MoleculeNet ESOL)

**Data kit lab** `moleculenet-esol-xgboost-rmse` · Discovery · about 20 minutes

**What this lab is.** It reproduces a benchmark's published model score. It is not a measurement and not a new model. You rebuild one number from the MoleculeNet benchmark paper (Wu and others, Chem Sci 2018): the test error of a gradient-boosted tree model (XGBoost) on fingerprints for the ESOL solubility set, printed in its Table 3 as an RMSE of 0.99 log units.

**The question.** How close does XGBoost on ECFP4 fingerprints get to that published test error, with the same data, the same split rule and the settings the benchmark's own code ships?

**The data.** `delaney-processed.csv` is the DeepChem project's own processed file for the MoleculeNet ESOL benchmark (DeepChem publishes MoleculeNet). It is not the supplement of Delaney's 2004 paper, and it is not the solubility set of the Therapeutics Data Commons. Licence: MIT, DeepChem repository, Copyright 2017 PandeLab.

**What you do.**

1. Load the file and look at the measured values.
2. Turn each structure into a 1,024-bit ECFP4 fingerprint with RDKit.
3. Split the compounds at random, 80/10/10, three times with fixed seeds.
4. Train XGBoost the way DeepChem 1.3.0 did, on each training subset.
5. Compute the test RMSE of each run and their mean, and compare the mean with the paper.

Where each setting comes from (the paper's text, DeepChem 1.3.0 code, or a choice made here) is written in this lab's SPEC.txt.

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'moleculenet-esol-xgboost-rmse'
FILES = json.loads(r'''{
 "esol": {
  "url": "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv",
  "name": "esol.csv",
  "sha256": "8c06a76f0c6487d29ab0f903e6a7a7139f189ab3c1178f159c8be8964602f189",
  "bytes": 96699
 }
}''')
PUBLISHED = json.loads(r'''{
 "test_rmse_xgboost_mean": {
  "label": "Test RMSE, XGBoost on ECFP4, mean of 3 random splits",
  "value": 0.99,
  "tolerance": {
   "abs": 0.108
  }
 },
 "n_compounds": {
  "label": "Compounds featurised and used",
  "value": 1128,
  "tolerance": {
   "exact": true
  }
 }
}''')
PUBLISHED_FROM = 'Europe PMC full-text record PMC5868307 (XML, no page numbers): Table 3, Summary of performances (test subset), row ESOL, metric RMSE, column Best performances - conventional methods; Table 1, row ESOL | https://www.ebi.ac.uk/europepmc/webservices/rest/PMC5868307/fullTextXML'
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. Libraries

The paper used DeepChem's code of 2017. Here the same steps are written out with current RDKit, scikit-learn and xgboost, so print the versions you ran.

In [ ]:
import numpy as np
import pandas as pd
import rdkit
import sklearn
import xgboost
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdMolDescriptors
from sklearn.model_selection import GridSearchCV, train_test_split
from xgboost import XGBRegressor

RDLogger.DisableLog("rdApp.*")  # silences RDKit's deprecation notice for the 2017-style fingerprint call
print("rdkit", rdkit.__version__, "| scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__)

## 2. The data

Two columns matter: `smiles` (the structure) and `measured log solubility in mols per litre` (log10 of mol/L, the value to predict). The file also holds a column of solubility predicted by the ESOL equation. It is not used for training; it is shown at the end for comparison.

In [ ]:
Y_COL = "measured log solubility in mols per litre"
ESOL_COL = "ESOL predicted log solubility in mols per litre"

df = pd.read_csv(data_file("esol"))
print("rows:", len(df), "| columns:", len(df.columns))
print(df[Y_COL].describe().round(2).to_string())
df[["Compound ID", "smiles", Y_COL]].head()

## 3. Fingerprints

The paper says it uses "ECFP4 fingerprints generated by RDKit". In DeepChem 1.3.0 that means a Morgan fingerprint of radius 2, folded to 1,024 bits, with bond types and without chirality. A structure RDKit cannot read is dropped and counted.

In [ ]:
def ecfp4_1024(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(
        mol, 2, nBits=1024, useChirality=False, useBondTypes=True, useFeatures=False)
    arr = np.zeros((1024,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

fps = [ecfp4_1024(s) for s in df["smiles"]]
keep = np.array([f is not None for f in fps])
print("structures RDKit could not read:", int((~keep).sum()))
data = df[keep].reset_index(drop=True)
X = np.vstack([f for f in fps if f is not None])
y = data[Y_COL].to_numpy(dtype=float)
n = len(y)
print("compounds featurised:", n, "| fingerprint bits:", X.shape[1], "| mean bits set per compound:", round(float(X.sum(1).mean()), 1))

## 4. Scale the measured values

DeepChem 1.3.0 subtracts the mean and divides by the standard deviation of the measured values over the whole file, before it splits. Errors are reported after the values are put back, in log units. Some XGBoost settings (max_delta_step, reg_lambda) act on this scaled value, so the scaling is kept.

In [ ]:
y_mean, y_sd = y.mean(), y.std()
z = (y - y_mean) / y_sd
print(f"mean {y_mean:.3f}, sd {y_sd:.3f} log units")

## 5. Random 80/10/10 splits

The paper splits every set 80/10/10 into training, validation and test subsets, and averages three runs with different random seeds. It does not print its seeds, so this lab fixes seeds 0, 1 and 2. The cut-offs follow DeepChem 1.3.0's RandomSplitter.

In [ ]:
SEEDS = (0, 1, 2)

def random_split(n, seed):
    np.random.seed(seed)
    perm = np.random.permutation(n)
    a, b = int(0.8 * n), int(0.9 * n)
    return perm[:a], perm[a:b], perm[b:]

for seed in SEEDS:
    tr, va, te = random_split(n, seed)
    print(f"seed {seed}: train {len(tr)}, valid {len(va)}, test {len(te)}")

## 6. The model, fitted as DeepChem 1.3.0 did

Starting values: DeepChem 1.3.0's preset for XGBoost regression. The paper also ran a brief hyperparameter search on the validation subset; its tuned values are in the paper's supplement, which this lab could not open. So the validation subset is not used for fitting here. It is scored and shown only.

The fit follows DeepChem 1.3.0's `XGBoostModel.fit`, on the training subset only:

1. A two-fold grid search with few, fast trees picks max_depth, colsample_bytree and reg_lambda.
2. An 80/20 split of the training subset finds the number of trees by early stopping.
3. The model is refitted on the whole training subset with 1.25 times that number of trees.

Current xgboost names: `seed` is now `random_state`, early stopping is set on the model, and `best_ntree_limit` is now `best_iteration + 1`.

In [ ]:
PRESET = dict(max_depth=5, learning_rate=0.05, n_estimators=3000, gamma=0.0, min_child_weight=5,
              max_delta_step=1, subsample=0.53, colsample_bytree=0.66, colsample_bylevel=1,
              reg_alpha=0, reg_lambda=1, scale_pos_weight=1, base_score=0.5,
              random_state=2016, n_jobs=4)
EARLY_STOPPING_ROUNDS = 100


def fit_like_deepchem_130(X_train, z_train):
    grid = {
        "max_depth": sorted({PRESET["max_depth"], 5, 7}),
        "colsample_bytree": sorted({PRESET["colsample_bytree"], 0.66, 0.9}),
        "reg_lambda": sorted({PRESET["reg_lambda"], 1, 5}),
        "learning_rate": [max(PRESET["learning_rate"], 0.3)],
        "n_estimators": [min(PRESET["n_estimators"], 60)],
    }
    search = GridSearchCV(XGBRegressor(**PRESET), grid, cv=2, refit=False,
                          scoring="neg_mean_absolute_error").fit(X_train, z_train)
    params = {**PRESET, **search.best_params_,
              "learning_rate": PRESET["learning_rate"], "n_estimators": PRESET["n_estimators"]}
    X_a, X_b, z_a, z_b = train_test_split(X_train, z_train, test_size=0.2, random_state=PRESET["random_state"])
    probe = XGBRegressor(**params, early_stopping_rounds=EARLY_STOPPING_ROUNDS, eval_metric="mae")
    probe.fit(X_a, z_a, eval_set=[(X_a, z_a), (X_b, z_b)], verbose=False)
    best_rounds = probe.best_iteration + 1
    params["n_estimators"] = int(np.round(best_rounds * 1.25))
    model = XGBRegressor(**params, eval_metric="mae").fit(X_train, z_train, verbose=False)
    chosen = {k: search.best_params_[k] for k in ("max_depth", "colsample_bytree", "reg_lambda")}
    return model, chosen, best_rounds, params["n_estimators"]


def rmse(pred_z, idx):
    pred = pred_z * y_sd + y_mean  # back to log10(mol/L)
    return float(np.sqrt(np.mean((pred - y[idx]) ** 2)))

## 7. Three runs

Each run takes a few seconds on a laptop CPU.

In [ ]:
runs = []
for seed in SEEDS:
    tr, va, te = random_split(n, seed)
    model, chosen, best_rounds, n_trees = fit_like_deepchem_130(X[tr], z[tr])
    runs.append({"seed": seed, **chosen, "early_stop_rounds": best_rounds, "trees": n_trees,
                 "rmse_train": rmse(model.predict(X[tr]), tr),
                 "rmse_valid": rmse(model.predict(X[va]), va),
                 "rmse_test": rmse(model.predict(X[te]), te),
                 "rmse_test_esol_column": float(np.sqrt(np.mean((data[ESOL_COL].to_numpy()[te] - y[te]) ** 2)))})
runs = pd.DataFrame(runs)
print(runs.round(3).to_string(index=False))
test_mean, test_sd = runs["rmse_test"].mean(), runs["rmse_test"].std(ddof=1)
print(f"\ntest RMSE, mean of {len(runs)} runs: {test_mean:.3f} (sd {test_sd:.3f}) log units")

## 8. For comparison: the ESOL equation's own predictions

The file carries the solubility predicted by the ESOL equation. Its error against the measured values needs no training. It is shown here, on all rows and on each test subset above, and is not checked against the paper.

In [ ]:
esol_all = float(np.sqrt(np.mean((data[ESOL_COL].to_numpy() - y) ** 2)))
print(f"RMSE of the file's ESOL column against measured, all {n} rows: {esol_all:.3f} log units")
print("on the three test subsets:", runs["rmse_test_esol_column"].round(3).tolist())
print("Table 3, graph-based column, for context only (not trained here): MPNN 0.58")

## 9. Compare with the paper

The check uses a tolerance of 0.108 log units, fixed in SPEC.txt before any run. It is two standard deviations of the difference between two means of three random splits with 113 test compounds each. It covers split-to-split chance only, not the paper's tuned hyperparameters or library versions.

In [ ]:
report({
    "test_rmse_xgboost_mean": float(test_mean),
    "n_compounds": int(n),
})

## What the result says

Read the line for the test RMSE. It says `match` if the mean of the three runs is within 0.108 of 0.99, and `DIFFERENT` if not. Then look at the three single runs in section 7. How far apart are they? Compare that spread with the gap to the paper.

Remember what was not reproduced: the paper's hyperparameter search and its seeds. A model score is a property of the data, the split, the seeds and the settings together. Quote all of them when you quote the number.

## Try next

1. Change `SEEDS` to five other seeds. How much does the mean move?
2. Change the fingerprint to 2,048 bits. That is a different featurisation from the paper's; does the score change by more than the spread between seeds?
3. Use the validation subset for what the paper used it for: try three values of `max_depth` and keep the one with the lowest validation RMSE. Then read the test RMSE once. Why must the test subset stay out of this choice?